In [1]:
## EXPOSURE 
import geopandas as gpd
import pandas as pd
from pathlib import Path
import shapely
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

from scipy.sparse import csr_matrix
from datetime import datetime

# on climada_petals branch feature/networks until merged!
# on climada_python develop branch
from climada.util import coordinates as u_coords
from climada.entity.exposures.base import Exposures
from climada.entity.impact_funcs import ImpactFunc, ImpactFuncSet
from climada.engine import Impact
from climada.hazard.base import Hazard
from climada.util import lines_polys_handler as u_lp
from climada.util.api_client import Client

#from climada_petals.entity.exposures.openstreetmap import osm_dataloader as osm
from climada_petals.entity.exposures import osm_dataloader as osm
from climada.util import coordinates as u_coords
from climada.hazard.base import Hazard

/home/gespejogutierrez/mambaforge/envs/climada_env/lib/python3.10/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(


In [2]:
import pickle

impbuilding = "/home/gespejogutierrez/Lisflood_climada/picklefiles/impbuilding_Zell_forecast.pkl"

with open(impbuilding, "rb") as f:
    impbuilding = pickle.load(f)

print("✔ Loaded impbuilding")
print(type(impbuilding))

✔ Loaded impbuilding
<class 'dict'>


In [3]:
impbuilding ['ZELL_2022-05-05T12:00'].imp_mat[10]

<1x1568 sparse matrix of type '<class 'numpy.float64'>'
	with 0 stored elements in Compressed Sparse Row format>

In [4]:
import geopandas as gpd
import xarray as xr
from shapely.geometry import box

BULD_SHP   = "/home/gespejogutierrez/Lisflood_climada/Data_for_process/geo_hwr_footp_tlm2023.shp"
HAZARD_ZELL = "/home/gespejogutierrez/Lisflood_climada/Zell_netcdfiles/Zell_2m_COSMO_2022-05-05T12-00.nc"

# --- 1) Read buildings ---
gdf_build = gpd.read_file(BULD_SHP)
if gdf_build.empty:
    raise ValueError("No polygons found in the shapefile.")

# Ensure CRS is set (LV95 / EPSG:2056 typical)
if gdf_build.crs is None:
    gdf_build = gdf_build.set_crs(2056, allow_override=True)
else:
    gdf_build = gdf_build.to_crs(2056)

# Rename active geometry column
gdf_build = gdf_build.rename_geometry("geom_polygon_lv95")

# --- 2) Get model domain bbox from NetCDF (LV95) ---
ds = xr.open_dataset(HAZARD_ZELL)
xmin, xmax = float(ds["x"].min()), float(ds["x"].max())
ymin, ymax = float(ds["y"].min()), float(ds["y"].max())

# Optional buffer in meters (recommended to avoid cutting edge effects)
buf = 200
xmin_b, xmax_b = xmin - buf, xmax + buf
ymin_b, ymax_b = ymin - buf, ymax + buf

domain_poly = box(xmin_b, ymin_b, xmax_b, ymax_b)

# --- 3) Subset buildings to the domain (FAST bbox filter) ---
# Since geometry is now "geom_polygon_lv95", we set it active temporarily for spatial ops
gdf_build = gdf_build.set_geometry("geom_polygon_lv95")

gdf_build_sub = gdf_build.cx[xmin_b:xmax_b, ymin_b:ymax_b].copy()

print("Buildings before:", len(gdf_build))
print("Buildings after bbox subset:", len(gdf_build_sub))

# --- 4) (Optional) Exact clip to domain bbox polygon (slower, more exact) ---
# If you want to physically cut polygons at the boundary:
# domain_gdf = gpd.GeoDataFrame(geometry=[domain_poly], crs="EPSG:2056")
# gdf_build_sub = gpd.clip(gdf_build_sub, domain_gdf)

# Keep the subset as your main object
gdf_build = gdf_build_sub

# Done: gdf_build now contains only buildings in your model domain
gdf_build.head()


Buildings before: 2154741
Buildings after bbox subset: 1568


,id_def,objectid,geom_polygon_lv95
13166,627672,627672.0,"POLYGON ((2705726.421 1255506.534, 2705725.835..."
13168,627870,627870.0,"POLYGON ((2704016.482 1255528.546, 2704018.014..."
13205,629980,629980.0,"POLYGON ((2706124.285 1255760.375, 2706124.357..."
13270,633463,633463.0,"POLYGON ((2704434.993 1256241.039, 2704435.319..."
13301,635820,635820.0,"POLYGON ((2702715.312 1256554.763, 2702716.035..."


In [5]:
# 1) make inside-points as geometry (LV95)
gdf_build["geometry"] = gdf_build["geom_polygon_lv95"].centroid
gdf_build = gdf_build.set_geometry("geometry")

# 4) Reproject points to WGS84 for CLIMADA; also keep polygons in WGS84 if you want
gdf_build = gdf_build.to_crs(4326)
gdf_build["geom_polygon_wgs84"] = gdf_build["geom_polygon_lv95"].to_crs(4326)

In [6]:
gdf_build = gdf_build.set_geometry("geometry")
gdf_build

,id_def,objectid,geom_polygon_lv95,geometry,geom_polygon_wgs84
13166,627672,627672.0,"POLYGON ((2705726.421 1255506.534, 2705725.835...",POINT (8.84023 47.44187),"POLYGON ((8.84035 47.44186, 8.84034 47.44182, ..."
13168,627870,627870.0,"POLYGON ((2704016.482 1255528.546, 2704018.014...",POINT (8.81766 47.44228),"POLYGON ((8.81769 47.44233, 8.81771 47.44232, ..."
13205,629980,629980.0,"POLYGON ((2706124.285 1255760.375, 2706124.357...",POINT (8.84566 47.44403),"POLYGON ((8.84568 47.44408, 8.84568 47.44405, ..."
13270,633463,633463.0,"POLYGON ((2704434.993 1256241.039, 2704435.319...",POINT (8.82333 47.44868),"POLYGON ((8.82340 47.44867, 8.82340 47.44863, ..."
13301,635820,635820.0,"POLYGON ((2702715.312 1256554.763, 2702716.035...",POINT (8.80036 47.45169),"POLYGON ((8.80067 47.45176, 8.80068 47.45173, ..."
...,...,...,...,...,...
2144513,1279188,1279188.0,"POLYGON ((2702812.763 1256573.584, 2702810.857...",POINT (8.80180 47.45186),"POLYGON ((8.80197 47.45191, 8.80194 47.45189, ..."
2144565,1284180,1284180.0,"POLYGON ((2705057.558 1255776.143, 2705054.464...",POINT (8.83151 47.44439),"POLYGON ((8.83154 47.44439, 8.83150 47.44436, ..."
2144594,1283854,1283854.0,"POLYGON ((2704184.851 1255563.419, 2704183.195...",POINT (8.81978 47.44269),"POLYGON ((8.81993 47.44261, 8.81990 47.44259, ..."
2144595,1283958,1283958.0,"POLYGON ((2704401.541 1255113.069, 2704400.540...",POINT (8.82256 47.43855),"POLYGON ((8.82269 47.43853, 8.82268 47.43849, ..."


In [7]:
import numpy as np
import geopandas as gpd


def impact_to_gdf_wide(
    imp,
    col_prefix="ens",          # ens0, ens1, ...
    add_exp_id=True,
    gdf_build=None,            # must contain 'geom_polygon_lv95'
):
    """
    Wide GeoDataFrame for ONE Impact object (one forecast time).

    Output:
      - geometry  : building polygons from gdf_build['geom_polygon_lv95'] (EPSG:2056)
      - lat, lon  : CLIMADA exposure coordinates (EPSG:4326)
      - ens0, ens1, ... : impact for each event / ensemble member
    """

    if gdf_build is None:
        raise ValueError("gdf_build (with 'geom_polygon_lv95') must be provided.")

    if "geom_polygon_lv95" not in gdf_build.columns:
        raise ValueError("gdf_build must have a 'geom_polygon_lv95' column.")

    n_events, n_exp = imp.imp_mat.shape

    if len(gdf_build) != n_exp:
        raise ValueError(
            f"gdf_build has {len(gdf_build)} rows but impact has {n_exp} exposures. "
            "They must be in the same order."
        )

    # ---------- start from building polygons ----------
    gdf = gdf_build.copy()

    # make sure polygon column is the active geometry in LV95
    gdf = gpd.GeoDataFrame(
        gdf,
        geometry="geom_polygon_lv95",
        crs="EPSG:2056",
    )

    # ---------- add CLIMADA exposure coordinates (lat, lon) ----------
    lats = imp.coord_exp[:, 0]
    lons = imp.coord_exp[:, 1]

    if len(lats) != n_exp:
        raise ValueError("Number of coord_exp points does not match number of exposures.")

    gdf["lat"] = lats
    gdf["lon"] = lons

    # optional exp_id
    if add_exp_id and "exp_id" not in gdf.columns:
        gdf["exp_id"] = np.arange(n_exp, dtype=int)

    # ---------- impacts: one column per ensemble member ----------
    mat = imp.imp_mat.tocsr()
    for i in range(n_events):
        gdf[f"{col_prefix}{i}"] = mat.getrow(i).toarray().ravel()

    return gdf



In [8]:
# impbuilding: dict like {'ZELL_2022-05-05T19:00': Impact, ...}

gdf_wide = {}   # this will be your dict of GeoDataFrames

for time_key, imp in impbuilding.items():
    gdf_wide[time_key] = impact_to_gdf_wide(
        imp,
        col_prefix="ens",
        add_exp_id=True,
        gdf_build=gdf_build,   # your table with geom_polygon_lv95
    )

In [9]:
gdf_wide ['ZELL_2022-05-05T12:00']

,id_def,objectid,geom_polygon_lv95,geometry,geom_polygon_wgs84,lat,lon,exp_id,ens0,ens1,ens2,ens3,ens4,ens5,ens6,ens7,ens8,ens9,ens10
13166,627672,627672.0,"POLYGON ((2705726.421 1255506.534, 2705725.835...",POINT (8.84023 47.44187),"POLYGON ((8.84035 47.44186, 8.84034 47.44182, ...",47.441870,8.840233,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13168,627870,627870.0,"POLYGON ((2704016.482 1255528.546, 2704018.014...",POINT (8.81766 47.44228),"POLYGON ((8.81769 47.44233, 8.81771 47.44232, ...",47.442282,8.817663,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13205,629980,629980.0,"POLYGON ((2706124.285 1255760.375, 2706124.357...",POINT (8.84566 47.44403),"POLYGON ((8.84568 47.44408, 8.84568 47.44405, ...",47.444032,8.845660,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13270,633463,633463.0,"POLYGON ((2704434.993 1256241.039, 2704435.319...",POINT (8.82333 47.44868),"POLYGON ((8.82340 47.44867, 8.82340 47.44863, ...",47.448684,8.823325,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13301,635820,635820.0,"POLYGON ((2702715.312 1256554.763, 2702716.035...",POINT (8.80036 47.45169),"POLYGON ((8.80067 47.45176, 8.80068 47.45173, ...",47.451694,8.800357,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2144513,1279188,1279188.0,"POLYGON ((2702812.763 1256573.584, 2702810.857...",POINT (8.80180 47.45186),"POLYGON ((8.80197 47.45191, 8.80194 47.45189, ...",47.451855,8.801795,1563,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2144565,1284180,1284180.0,"POLYGON ((2705057.558 1255776.143, 2705054.464...",POINT (8.83151 47.44439),"POLYGON ((8.83154 47.44439, 8.83150 47.44436, ...",47.444390,8.831512,1564,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2144594,1283854,1283854.0,"POLYGON ((2704184.851 1255563.419, 2704183.195...",POINT (8.81978 47.44269),"POLYGON ((8.81993 47.44261, 8.81990 47.44259, ...",47.442693,8.819780,1565,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2144595,1283958,1283958.0,"POLYGON ((2704401.541 1255113.069, 2704400.540...",POINT (8.82256 47.43855),"POLYGON ((8.82269 47.43853, 8.82268 47.43849, ...",47.438551,8.822562,1566,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
###Plot ensemble agreement 

In [10]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import pyproj
import os
from datetime import datetime, timedelta


def plot_ensemble_agreement(
    gdf_for_time,
    time_key,
    *,
    threshold=0.03,
    dpi=300,
    outdir="ZELL_PLOTS",
    transparent_bg=True,
    xmin=None,  # LV95
    xmax=None,  # LV95
    ymin=None,  # LV95
    ymax=None,  # LV95
):
    """
    Plot ensemble agreement for one forecast time.

    - gdf_for_time: GeoDataFrame for a single time step (one entry of gdf_wide),
      geometry in LV95 (EPSG:2056).
    - xmin/xmax/ymin/ymax: bounding box in LV95; if not given, uses data extent.

    The background WMS is plotted in EPSG:3857 (Web Mercator) and the building
    polygons are reprojected from LV95 to EPSG:3857 to avoid any shift.
    """

    # ---------- 0) Ensure LV95 CRS on input ----------
    gdf_lv95 = gdf_for_time.copy()
    gdf_lv95 = gdf_lv95.set_crs(epsg=2056, allow_override=True)

    # -----------------------
    # 1) Identify ensemble columns
    # -----------------------
    ens_cols = [c for c in gdf_lv95.columns if c.startswith("ens")]
    n_ens = len(ens_cols)
    if n_ens == 0:
        print(f"No ensemble columns for {time_key} — skipping.")
        return

    # -----------------------
    # 2) Count ensembles >= threshold
    # -----------------------
    gdf_lv95["n_ens"] = (gdf_lv95[ens_cols] >= threshold).sum(axis=1).astype(int)

    subset_lv95 = gdf_lv95[gdf_lv95["n_ens"] > 0].copy()
    if subset_lv95.empty:
        print(f"No impacts ≥ {threshold} for {time_key}")
        return

    # -----------------------
    # 3) Determine LV95 bbox and convert to EPSG:3857 for plotting
    # -----------------------
    if None not in (xmin, xmax, ymin, ymax):
        subset_lv95 = subset_lv95.cx[xmin:xmax, ymin:ymax]
        if subset_lv95.empty:
            print(f"No impacted buildings inside bbox for {time_key}")
            return
    else:
        xmin, ymin, xmax, ymax = subset_lv95.total_bounds

    # transformer LV95 -> Web Mercator
    transformer = pyproj.Transformer.from_crs(2056, 3857, always_xy=True)
    xmin_m, ymin_m = transformer.transform(xmin, ymin)
    xmax_m, ymax_m = transformer.transform(xmax, ymax)
    extent_3857 = (xmin_m, xmax_m, ymin_m, ymax_m)

    # reproject polygons to EPSG:3857 for plotting
    subset_3857 = subset_lv95.to_crs(epsg=3857)

    proj_plot = ccrs.epsg(3857)  # Web Mercator

    # -----------------------
    # 4) Color bins: 1–3, 3–5, 5–7, 7–9, 9–11
    # -----------------------
    bounds = np.array([0.5, 3.5, 5.5, 7.5, 9.5, 11.5])
    base_cmap = plt.cm.magma
    cmap = mcolors.ListedColormap(base_cmap(np.linspace(0.25, 0.95, 5)))
    norm = mcolors.BoundaryNorm(bounds, cmap.N)

    # -----------------------
    # 5) Build title with local time (+2 h)
    # -----------------------
    parts = time_key.split("_", 1)
    prefix = parts[0]          # 'ZELL'
    utc_str = parts[1]         # '2022-05-05T19:00'

    try:
        dt_utc = datetime.fromisoformat(utc_str)
        dt_local = dt_utc + timedelta(hours=2)
        local_str = dt_local.strftime("%Y-%m-%dT%H:%M")
        title_text = f"{time_key}  (local time: {local_str})"
    except Exception:
        title_text = time_key

    # -----------------------
    # 6) Plot
    # -----------------------
    fig = plt.figure(figsize=(10, 10), dpi=dpi)
    ax = plt.axes(projection=proj_plot)
    ax.set_extent(extent_3857, crs=proj_plot)
    ax.set_aspect("1")

    if transparent_bg:
        fig.patch.set_alpha(0)
        ax.set_facecolor((0, 0, 0, 0))

    # Background WMS in Web Mercator
    ax.add_wms(
        "https://wms.geo.admin.ch/?",
        layers="ch.swisstopo.swisstlm3d-karte-grau",
        alpha=0.85,
        zorder=0,
    )

    # Polygons in EPSG:3857
    subset_3857.plot(
        ax=ax,
        column="n_ens",
        cmap=cmap,
        norm=norm,
        linewidth=0.2,
        edgecolor="black",
        alpha=0.9,
        zorder=5,
        transform=proj_plot,  # same CRS as axis
    )

    # Title with local time
    ax.set_title(
        title_text,
        fontsize=14,
        color="black",
        pad=8,
    )

    # Colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.036, pad=0.02)
    cbar.set_label("Number of ensemble members", fontsize=11)
    cbar.set_ticks([2, 4, 6, 8, 10])
    cbar.set_ticklabels(["1–3", "3–5", "5–7", "7–9", "9–11"])

    # -----------------------
    # 7) Save
    # -----------------------
    os.makedirs(outdir, exist_ok=True)
    tstr = utc_str.replace(":", "")

    fn = os.path.join(
        outdir,
        f"forecast_ensemble_{prefix}_{tstr}.png"
    )

    plt.savefig(fn, dpi=dpi, bbox_inches="tight", transparent=transparent_bg)
    plt.close(fig)
    print(f"✔ Saved {fn}")


In [13]:
outdir = "/home/gespejogutierrez/Lisflood_climada/Zell_forecast_plots"

for time_key, gdf_for_time in gdf_wide.items():
    plot_ensemble_agreement(
        gdf_for_time,
        time_key=time_key,
        threshold=0.03,
        dpi=300,
        outdir=outdir,
        xmin=2702800,
        xmax=2705600,
        ymin=1254750,
        ymax=1257270
    )

No impacts ≥ 0.03 for ZELL_2022-05-05T12:00
No impacts ≥ 0.03 for ZELL_2022-05-05T13:00
No impacts ≥ 0.03 for ZELL_2022-05-05T14:00
No impacted buildings inside bbox for ZELL_2022-05-05T15:00
No impacted buildings inside bbox for ZELL_2022-05-05T16:00
No impacted buildings inside bbox for ZELL_2022-05-05T17:00


/home/gespejogutierrez/mambaforge/envs/climada_env/lib/python3.10/site-packages/owslib/map/wms111.py:121: UserWarning: Content metadata for layer "ch.vbs.kataster-belasteter-standorte-militaer" already exists. Using child layer
  warnings.warn('Content metadata for layer "%s" already exists. Using child layer' % cm.id)
/home/gespejogutierrez/mambaforge/envs/climada_env/lib/python3.10/site-packages/owslib/map/wms111.py:121: UserWarning: Content metadata for layer "ch.vbs.kataster-belasteter-standorte-militaer_v2_0.oereb" already exists. Using child layer
  warnings.warn('Content metadata for layer "%s" already exists. Using child layer' % cm.id)
/home/gespejogutierrez/mambaforge/envs/climada_env/lib/python3.10/site-packages/owslib/map/wms111.py:121: UserWarning: Content metadata for layer "ch.vbs.panzerverschiebungsrouten" already exists. Using child layer
  warnings.warn('Content metadata for layer "%s" already exists. Using child layer' % cm.id)
/home/gespejogutierrez/mambaforge/envs/

✔ Saved /home/gespejogutierrez/Lisflood_climada/Zell_forecast_plots/forecast_ensemble_ZELL_2022-05-05T1800.png


/home/gespejogutierrez/mambaforge/envs/climada_env/lib/python3.10/site-packages/owslib/map/wms111.py:121: UserWarning: Content metadata for layer "ch.vbs.kataster-belasteter-standorte-militaer" already exists. Using child layer
  warnings.warn('Content metadata for layer "%s" already exists. Using child layer' % cm.id)
/home/gespejogutierrez/mambaforge/envs/climada_env/lib/python3.10/site-packages/owslib/map/wms111.py:121: UserWarning: Content metadata for layer "ch.vbs.kataster-belasteter-standorte-militaer_v2_0.oereb" already exists. Using child layer
  warnings.warn('Content metadata for layer "%s" already exists. Using child layer' % cm.id)
/home/gespejogutierrez/mambaforge/envs/climada_env/lib/python3.10/site-packages/owslib/map/wms111.py:121: UserWarning: Content metadata for layer "ch.vbs.panzerverschiebungsrouten" already exists. Using child layer
  warnings.warn('Content metadata for layer "%s" already exists. Using child layer' % cm.id)
/home/gespejogutierrez/mambaforge/envs/

✔ Saved /home/gespejogutierrez/Lisflood_climada/Zell_forecast_plots/forecast_ensemble_ZELL_2022-05-05T1900.png


...

✔ Saved /home/gespejogutierrez/Lisflood_climada/Zell_forecast_plots/forecast_ensemble_ZELL_2022-05-05T2000.png


/home/gespejogutierrez/mambaforge/envs/climada_env/lib/python3.10/site-packages/owslib/map/wms111.py:121: UserWarning: Content metadata for layer "ch.vbs.kataster-belasteter-standorte-militaer" already exists. Using child layer
  warnings.warn('Content metadata for layer "%s" already exists. Using child layer' % cm.id)
/home/gespejogutierrez/mambaforge/envs/climada_env/lib/python3.10/site-packages/owslib/map/wms111.py:121: UserWarning: Content metadata for layer "ch.vbs.kataster-belasteter-standorte-militaer_v2_0.oereb" already exists. Using child layer
  warnings.warn('Content metadata for layer "%s" already exists. Using child layer' % cm.id)
/home/gespejogutierrez/mambaforge/envs/climada_env/lib/python3.10/site-packages/owslib/map/wms111.py:121: UserWarning: Content metadata for layer "ch.vbs.panzerverschiebungsrouten" already exists. Using child layer
  warnings.warn('Content metadata for layer "%s" already exists. Using child layer' % cm.id)
/home/gespejogutierrez/mambaforge/envs/

✔ Saved /home/gespejogutierrez/Lisflood_climada/Zell_forecast_plots/forecast_ensemble_ZELL_2022-05-05T2100.png


/home/gespejogutierrez/mambaforge/envs/climada_env/lib/python3.10/site-packages/owslib/map/wms111.py:121: UserWarning: Content metadata for layer "ch.vbs.kataster-belasteter-standorte-militaer" already exists. Using child layer
  warnings.warn('Content metadata for layer "%s" already exists. Using child layer' % cm.id)
/home/gespejogutierrez/mambaforge/envs/climada_env/lib/python3.10/site-packages/owslib/map/wms111.py:121: UserWarning: Content metadata for layer "ch.vbs.kataster-belasteter-standorte-militaer_v2_0.oereb" already exists. Using child layer
  warnings.warn('Content metadata for layer "%s" already exists. Using child layer' % cm.id)
/home/gespejogutierrez/mambaforge/envs/climada_env/lib/python3.10/site-packages/owslib/map/wms111.py:121: UserWarning: Content metadata for layer "ch.vbs.panzerverschiebungsrouten" already exists. Using child layer
  warnings.warn('Content metadata for layer "%s" already exists. Using child layer' % cm.id)
/home/gespejogutierrez/mambaforge/envs/

✔ Saved /home/gespejogutierrez/Lisflood_climada/Zell_forecast_plots/forecast_ensemble_ZELL_2022-05-05T2200.png


In [16]:
gdf_wide ['ZELL_2022-05-05T17:00']

,id_def,objectid,geom_polygon_lv95,geometry,geom_polygon_wgs84,lat,lon,exp_id,ens0,ens1,ens2,ens3,ens4,ens5,ens6,ens7,ens8,ens9,ens10
13166,627672,627672.0,"POLYGON ((2705726.421 1255506.534, 2705725.835...",POINT (8.84023 47.44187),"POLYGON ((8.84035 47.44186, 8.84034 47.44182, ...",47.441870,8.840233,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13168,627870,627870.0,"POLYGON ((2704016.482 1255528.546, 2704018.014...",POINT (8.81766 47.44228),"POLYGON ((8.81769 47.44233, 8.81771 47.44232, ...",47.442282,8.817663,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13205,629980,629980.0,"POLYGON ((2706124.285 1255760.375, 2706124.357...",POINT (8.84566 47.44403),"POLYGON ((8.84568 47.44408, 8.84568 47.44405, ...",47.444032,8.845660,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13270,633463,633463.0,"POLYGON ((2704434.993 1256241.039, 2704435.319...",POINT (8.82333 47.44868),"POLYGON ((8.82340 47.44867, 8.82340 47.44863, ...",47.448684,8.823325,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13301,635820,635820.0,"POLYGON ((2702715.312 1256554.763, 2702716.035...",POINT (8.80036 47.45169),"POLYGON ((8.80067 47.45176, 8.80068 47.45173, ...",47.451694,8.800357,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2144513,1279188,1279188.0,"POLYGON ((2702812.763 1256573.584, 2702810.857...",POINT (8.80180 47.45186),"POLYGON ((8.80197 47.45191, 8.80194 47.45189, ...",47.451855,8.801795,1563,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2144565,1284180,1284180.0,"POLYGON ((2705057.558 1255776.143, 2705054.464...",POINT (8.83151 47.44439),"POLYGON ((8.83154 47.44439, 8.83150 47.44436, ...",47.444390,8.831512,1564,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2144594,1283854,1283854.0,"POLYGON ((2704184.851 1255563.419, 2704183.195...",POINT (8.81978 47.44269),"POLYGON ((8.81993 47.44261, 8.81990 47.44259, ...",47.442693,8.819780,1565,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2144595,1283958,1283958.0,"POLYGON ((2704401.541 1255113.069, 2704400.540...",POINT (8.82256 47.43855),"POLYGON ((8.82269 47.43853, 8.82268 47.43849, ...",47.438551,8.822562,1566,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [17]:
import numpy as np
import pandas as pd
import geopandas as gpd


def impact_to_gdf_long(
    imp,
    time_key=None,              # e.g. "ZELL_2022-05-05T12:00" (optional)
    gdf_build=None,             # must contain 'geom_polygon_lv95' (and ideally 'exp_id')
    exp_id_col="exp_id",
    geom_col="geom_polygon_lv95",
    event_name_prefix="ens",    # if imp.event_name missing, fallback to ens0, ens1, ...
    keep_point_geometry=False,  # add exposure point geometry (WGS84) as extra column
):
    """
    LONG GeoDataFrame for ONE Impact object (one forecast time).

    Output columns (typical):
      - geometry      : building polygons from gdf_build[geom_col] (EPSG:2056)
      - lat, lon      : CLIMADA exposure coordinates (EPSG:4326)
      - exp_id        : exposure id (from gdf_build if present, else generated)
      - event_id      : 1..n_events
      - event_name    : CLIMADA event name (if available) or ens0, ens1, ...
      - impact        : impact value for (event, exposure)
      - time_key      : your passed key (optional)
      - run_datetime  : parsed datetime from time_key if possible (optional)
    """

    if gdf_build is None:
        raise ValueError("gdf_build (with polygons) must be provided.")
    if geom_col not in gdf_build.columns:
        raise ValueError(f"gdf_build must have a '{geom_col}' column.")

    n_events, n_exp = imp.imp_mat.shape

    # Ensure same number of exposures as polygons table
    if len(gdf_build) != n_exp:
        raise ValueError(
            f"gdf_build has {len(gdf_build)} rows but impact has {n_exp} exposures. "
            "They must match and be in the same order."
        )

    # Base table from polygons
    base = gdf_build.copy()

    # Ensure exp_id exists (stable join key)
    if exp_id_col not in base.columns:
        base[exp_id_col] = np.arange(n_exp, dtype=int)

    # Set polygon geometry (LV95)
    base = gpd.GeoDataFrame(base, geometry=geom_col, crs="EPSG:2056")

    # Add exposure coords (WGS84)
    lats = imp.coord_exp[:, 0]
    lons = imp.coord_exp[:, 1]
    if len(lats) != n_exp:
        raise ValueError("Number of coord_exp points does not match number of exposures.")

    base["lat"] = lats
    base["lon"] = lons

    # Optional: keep exposure point geometry as extra column
    if keep_point_geometry:
        base["geometry_wgs84_pt"] = gpd.points_from_xy(base["lon"], base["lat"], crs="EPSG:4326")

    # Event labels
    if hasattr(imp, "event_name") and imp.event_name is not None and len(imp.event_name) == n_events:
        event_names = list(imp.event_name)
    else:
        event_names = [f"{event_name_prefix}{i}" for i in range(n_events)]

    # Convert sparse -> COO for efficient long format
    mat = imp.imp_mat.tocoo()

    # Create long rows ONLY where impact is non-zero
    df_long = pd.DataFrame({
        "event_idx": mat.row.astype(int),       # 0-based
        "exp_idx":   mat.col.astype(int),       # 0-based
        "impact":    mat.data.astype(float),
    })

    # Map event fields
    df_long["event_id"] = df_long["event_idx"] + 1
    df_long["event_name"] = [event_names[i] for i in df_long["event_idx"].values]

    # Attach exposure ids and coordinates by exp_idx
    df_long[exp_id_col] = base[exp_id_col].values[df_long["exp_idx"].values]
    df_long["lat"] = base["lat"].values[df_long["exp_idx"].values]
    df_long["lon"] = base["lon"].values[df_long["exp_idx"].values]

    # Optional time info
    if time_key is not None:
        df_long["time_key"] = str(time_key)
        # try to parse datetime from key like "ZELL_2022-05-05T12:00"
        try:
            dt_str = str(time_key).split("_", 1)[-1]
            df_long["run_datetime"] = pd.to_datetime(dt_str)
        except Exception:
            pass

    # Now build GeoDataFrame by joining polygons from base using exp_idx
    # We avoid a merge by using direct indexing (fast and preserves order)
    geom = base.geometry.values[df_long["exp_idx"].values]
    gdf_long = gpd.GeoDataFrame(df_long, geometry=geom, crs="EPSG:2056")

    # Clean helper cols if you don't want them
    gdf_long = gdf_long.drop(columns=["event_idx", "exp_idx"])

    return gdf_long


In [18]:
# Example usage with your dict of impacts
# -----------------------
gdf_long = {}
for time_key, imp in impbuilding.items():
     gdf_long[time_key] = impact_to_gdf_long(
         imp,
         time_key=time_key,
        gdf_build=gdf_build,
         exp_id_col="exp_id",
         geom_col="geom_polygon_lv95",
         keep_point_geometry=False
     )

In [21]:
gdf_long ['ZELL_2022-05-05T18:00']

,impact,event_id,event_name,exp_id,lat,lon,time_key,run_datetime,geometry
0,0.037604,1,ZELL_ens01_2022-05-05T18:00,48,47.435405,8.855446,ZELL_2022-05-05T18:00,2022-05-05 18:00:00,"POLYGON ((2706872.658 1254811.771, 2706872.916..."
1,0.022500,1,ZELL_ens01_2022-05-05T18:00,331,47.435957,8.849880,ZELL_2022-05-05T18:00,2022-05-05 18:00:00,"POLYGON ((2706464.410 1254872.769, 2706463.182..."
2,0.010500,1,ZELL_ens01_2022-05-05T18:00,341,47.436270,8.855717,ZELL_2022-05-05T18:00,2022-05-05 18:00:00,"POLYGON ((2706889.050 1254901.000, 2706886.350..."
3,0.029250,1,ZELL_ens01_2022-05-05T18:00,414,47.435037,8.856905,ZELL_2022-05-05T18:00,2022-05-05 18:00:00,"POLYGON ((2706993.785 1254777.219, 2706994.061..."
4,0.015000,1,ZELL_ens01_2022-05-05T18:00,804,47.436959,8.851021,ZELL_2022-05-05T18:00,2022-05-05 18:00:00,"POLYGON ((2706546.483 1254979.245, 2706545.512..."
...,...,...,...,...,...,...,...,...,...
559,0.023250,11,ZELL_ens11_2022-05-05T18:00,1498,47.438281,8.845481,ZELL_2022-05-05T18:00,2022-05-05 18:00:00,"POLYGON ((2706118.065 1255125.876, 2706119.731..."
560,0.031562,11,ZELL_ens11_2022-05-05T18:00,1519,47.437556,8.846585,ZELL_2022-05-05T18:00,2022-05-05 18:00:00,"POLYGON ((2706210.835 1255036.980, 2706207.552..."
561,0.018000,11,ZELL_ens11_2022-05-05T18:00,1520,47.437162,8.849718,ZELL_2022-05-05T18:00,2022-05-05 18:00:00,"POLYGON ((2706451.224 1255000.998, 2706450.262..."
562,0.008250,11,ZELL_ens11_2022-05-05T18:00,1546,47.464452,8.846096,ZELL_2022-05-05T18:00,2022-05-05 18:00:00,"POLYGON ((2706123.426 1258032.800, 2706121.851..."


In [ ]:
### How many are in the Gemeinde Zell 